In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from src.to_html import cell, heading, paragraph, table

In [2]:
data_dir = Path("../monthly_processed_data")
files = sorted(data_dir.glob("data-*.parquet"))[-2:]

df_list = []
for file in files:
    df_list.append(pd.read_parquet(file)[["station_name", "train_type"]])
df = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(df):,} records from {len(files)} files")
print(f"Files: {[f.name for f in files]}")

Loaded 31,046,215 records from 2 files
Files: ['data-2025-12.parquet', 'data-2026-01.parquet']


In [3]:
# Function to categorize train types
def categorize_train_type(train_type):
    return train_type if train_type in ["IC", "ICE", "RB", "RE", "S"] else "Sonstige"


# Add a new column for categorized train types
df["train_type_category"] = df["train_type"].map(categorize_train_type)

# Group by station and train type, then count
station_train_counts = df.groupby(["station_name", "train_type_category"]).size().unstack(fill_value=0)

# Add total column
station_train_counts["Total"] = station_train_counts.sum(axis=1)

# Sort by ICE count descending
station_train_counts.sort_values(by="ICE", ascending=False, inplace=True)

# Prepare DataFrame for display
result_df = station_train_counts.reset_index()
result_df.columns = ["Bahnhof", "IC", "ICE", "RB", "RE", "S", "Sonstige", "Alle Züge"]
result_df = result_df[["Bahnhof", "Alle Züge", "ICE", "IC", "RE", "RB", "S", "Sonstige"]]

content = cell(
    heading("An welchem Bahnhof halten die meisten ICE, RE oder andere Zuggattungen?", level=1)
    + paragraph(
        "In der Tabelle kann man alle Bahnhöfe sehen mit der Anzahl an Zughalten nach den üblichen Zuggattungen aufgeteilt."
    )
    + paragraph(
        "In manchen Regionen gibt es keine REs aber dafür regionale äquivalente Zuggattungen. "
        'In Lüneburg zum Beispiel gibt es keine RE, RB oder S Bahnen, aber dafür "ME" und "erx" Züge. '
        'Um genau zu sehen welche Zuggattungen an einem Bahnhof halten gibt es <a href="verspaetung_pro_bahnhof.html">Verspätungen pro Bahnhof</a>.'
    )
    + table(result_df)
)
display(HTML(content))

Bahnhof ⇕,Alle Züge ⇕,ICE ⇕,IC ⇕,RE ⇕,RB ⇕,S ⇕,Sonstige ⇕
Berlin Hauptbahnhof,101091,15947,1004,23862,902,45769,13607
Frankfurt (Main) Hbf,125351,15231,733,15431,12998,58977,21981
München Hbf,133388,12325,285,9342,14496,66506,30434
Hannover Hbf,58808,11783,2323,2726,2325,5471,34180
Berlin-Spandau,37172,10638,26,5254,2546,14409,4299
Frankfurt am Main Flughafen Fernbahnhof,13297,10429,178,0,0,152,2538
Hamburg Hbf,121024,9832,676,11882,5183,75303,18148
Nürnberg Hbf,66048,9690,715,18443,6547,27202,3451
Köln Hbf,75511,9490,1888,9265,13249,26904,14715
Mannheim Hbf,41293,8644,895,9770,613,20177,1194


In [4]:
start_date = files[0].stem.replace("data-", "")
end_date = files[-1].stem.replace("data-", "")
content = cell(
    f'<p style="font-size: 0.8em; text-align: center;">Quelle: <a href="https://github.com/piebro/deutsche-bahn-data/blob/main/notebooks/zuggattungen_pro_bahnhof.ipynb">Berechnet</a> auf Basis von <a href="https://github.com/piebro/deutsche-bahn-data">gesammelten Daten</a> von der Deutschen Bahn vom {start_date} bis {end_date}.</p>'
)
display(HTML(content))